<a href="https://colab.research.google.com/github/xiyuan1avery/ma2288/blob/research-v2/notebooks/11_v2_acceptance_setup.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q transformers

In [2]:
import random
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Device: cuda
GPU: Tesla T4


In [3]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [4]:
DRIVE_DIRECTORY = Path(
    "/content/drive/MyDrive/ma2288_nextlat"
)

TEST_DATA_PATH = (
    DRIVE_DIRECTORY
    / "test_teacher_states.pt"
)

ONE_STEP_CHECKPOINT_PATH = (
    DRIVE_DIRECTORY
    / "checkpoints"
    / "one_step_transition_seed42.pt"
)

MULTISTEP_CHECKPOINT_PATH = (
    DRIVE_DIRECTORY
    / "checkpoints"
    / "multistep_transition_seed42.pt"
)

print("Test data:", TEST_DATA_PATH.exists())
print(
    "One-step checkpoint:",
    ONE_STEP_CHECKPOINT_PATH.exists(),
)
print(
    "Multi-step checkpoint:",
    MULTISTEP_CHECKPOINT_PATH.exists(),
)

Test data: True
One-step checkpoint: True
Multi-step checkpoint: True


In [5]:
test_artifact = torch.load(
    TEST_DATA_PATH,
    map_location="cpu",
    weights_only=False,
)

test_token_ids = test_artifact[
    "token_ids"
]

MODEL_NAME = test_artifact[
    "model_name"
]

print("Model:", MODEL_NAME)
print("Test tokens:", test_token_ids.shape)

Model: distilgpt2
Test tokens: torch.Size([80, 64])


In [6]:
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

target_model = (
    AutoModelForCausalLM
    .from_pretrained(MODEL_NAME)
    .to(device)
)

target_model.eval()

for parameter in target_model.parameters():
    parameter.requires_grad = False

token_embedding = (
    target_model.get_input_embeddings()
)

print("Target model loaded.")
print(
    "Vocabulary size:",
    target_model.config.vocab_size,
)

config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  353MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Target model loaded.
Vocabulary size: 50257


In [7]:
class ResidualTransitionMLP(nn.Module):
    def __init__(
        self,
        hidden_dimension=768,
        bottleneck_dimension=512,
    ):
        super().__init__()

        combined_dimension = (
            hidden_dimension * 2
        )

        self.input_normalization = nn.LayerNorm(
            combined_dimension
        )

        self.network = nn.Sequential(
            nn.Linear(
                combined_dimension,
                bottleneck_dimension,
            ),
            nn.GELU(),
            nn.Linear(
                bottleneck_dimension,
                hidden_dimension,
            ),
        )

    def forward(
        self,
        current_hidden,
        next_token_embedding,
    ):
        combined_input = torch.cat(
            [
                current_hidden,
                next_token_embedding,
            ],
            dim=-1,
        )

        normalized_input = (
            self.input_normalization(
                combined_input
            )
        )

        predicted_change = self.network(
            normalized_input
        )

        return (
            current_hidden
            + predicted_change
        )

In [8]:
def load_transition_model(
    checkpoint_path,
):
    checkpoint = torch.load(
        checkpoint_path,
        map_location="cpu",
        weights_only=False,
    )

    model = ResidualTransitionMLP(
        hidden_dimension=checkpoint[
            "hidden_dimension"
        ],
        bottleneck_dimension=checkpoint[
            "bottleneck_dimension"
        ],
    )

    model.load_state_dict(
        checkpoint["model_state_dict"]
    )

    model = model.to(device)
    model.eval()

    return model, checkpoint

In [9]:
(
    one_step_model,
    one_step_checkpoint,
) = load_transition_model(
    ONE_STEP_CHECKPOINT_PATH
)

(
    multistep_model,
    multistep_checkpoint,
) = load_transition_model(
    MULTISTEP_CHECKPOINT_PATH
)

print("Both transition models loaded.")

Both transition models loaded.


In [10]:
PREFIX_LENGTH = 32
PROMPT_INDEX = 0

prefix_ids = (
    test_token_ids[
        PROMPT_INDEX,
        :PREFIX_LENGTH,
    ]
    .unsqueeze(0)
    .to(device)
)

print("Prefix shape:", prefix_ids.shape)

print("\nDecoded prefix:\n")
print(
    tokenizer.decode(
        prefix_ids[0]
    )
)

Prefix shape: torch.Size([1, 32])

Decoded prefix:

= Robert Boulter =<|endoftext|>Robert Boulter is an English film , television and theatre actor . He had a guest @-@ starring role on the


In [11]:
with torch.no_grad():
    prefix_outputs = target_model(
        input_ids=prefix_ids,
        output_hidden_states=True,
        return_dict=True,
    )

true_prefix_hidden = (
    prefix_outputs.hidden_states[-1][
        :,
        -1,
        :,
    ]
)

target_first_logits = (
    prefix_outputs.logits[
        :,
        -1,
        :,
    ]
)

direct_head_logits = (
    target_model.lm_head(
        true_prefix_hidden
    )
)

maximum_logit_difference = (
    torch.max(
        torch.abs(
            target_first_logits
            - direct_head_logits
        )
    )
    .item()
)

print(
    "Maximum logit difference:",
    maximum_logit_difference,
)

Maximum logit difference: 5.340576171875e-05


In [12]:
def logits_to_probabilities(logits):
    return torch.softmax(
        logits,
        dim=-1,
    )

In [13]:
q1 = logits_to_probabilities(
    direct_head_logits
)

torch.manual_seed(SEED)

draft_token_1 = torch.multinomial(
    q1,
    num_samples=1,
)

print(
    "Draft token 1 ID:",
    draft_token_1.item(),
)

print(
    "Draft token 1 text:",
    repr(
        tokenizer.decode(
            draft_token_1[0]
        )
    ),
)

Draft token 1 ID: 2277
Draft token 1 text: ' hit'


In [14]:
with torch.no_grad():
    token_1_embedding = token_embedding(
        draft_token_1
    ).squeeze(1)

    predicted_hidden_1 = (
        multistep_model(
            true_prefix_hidden,
            token_1_embedding,
        )
    )

    predicted_logits_2 = (
        target_model.lm_head(
            predicted_hidden_1
        )
    )

    q2 = logits_to_probabilities(
        predicted_logits_2
    )

In [15]:
draft_token_2 = torch.multinomial(
    q2,
    num_samples=1,
)

print(
    "Draft token 2 ID:",
    draft_token_2.item(),
)

print(
    "Draft token 2 text:",
    repr(
        tokenizer.decode(
            draft_token_2[0]
        )
    ),
)

Draft token 2 ID: 262
Draft token 2 text: ' the'


In [16]:
verification_input = torch.cat(
    [
        prefix_ids,
        draft_token_1,
    ],
    dim=1,
)

with torch.no_grad():
    verification_outputs = target_model(
        input_ids=verification_input,
        return_dict=True,
    )

target_logits_2 = (
    verification_outputs.logits[
        :,
        -1,
        :,
    ]
)

p2 = logits_to_probabilities(
    target_logits_2
)

print(
    "Verification input length:",
    verification_input.shape[1],
)

Verification input length: 33


In [17]:
log_p2 = torch.log(
    p2.clamp_min(1e-12)
)

log_q2 = torch.log(
    q2.clamp_min(1e-12)
)

output_kl_step_2 = torch.sum(
    p2 * (log_p2 - log_q2),
    dim=-1,
)

print(
    "Step-2 output KL:",
    output_kl_step_2.item(),
)

Step-2 output KL: 1.4036126136779785


In [18]:
draft_token_2_probability_under_p = (
    p2.gather(
        dim=1,
        index=draft_token_2,
    )
    .squeeze()
)

draft_token_2_probability_under_q = (
    q2.gather(
        dim=1,
        index=draft_token_2,
    )
    .squeeze()
)

acceptance_probability_step_2 = (
    torch.minimum(
        torch.ones(
            (),
            device=device,
        ),
        (
            draft_token_2_probability_under_p
            / draft_token_2_probability_under_q
              .clamp_min(1e-12)
        ),
    )
)

print(
    "p(draft token 2):",
    draft_token_2_probability_under_p.item(),
)

print(
    "q(draft token 2):",
    draft_token_2_probability_under_q.item(),
)

print(
    "Step-2 acceptance probability:",
    acceptance_probability_step_2.item(),
)

p(draft token 2): 0.0003640451468527317
q(draft token 2): 0.04324853792786598
Step-2 acceptance probability: 0.008417513221502304


In [19]:
uniform_random_value = torch.rand(
    (),
    device=device,
)

accepted_step_2 = (
    uniform_random_value
    < acceptance_probability_step_2
)

print(
    "Uniform random value:",
    uniform_random_value.item(),
)

print(
    "Accepted step 2:",
    accepted_step_2.item(),
)

Uniform random value: 0.8258668184280396
Accepted step 2: False


In [20]:
def compute_step_2_kl(
    transition_model,
):
    with torch.no_grad():
        predicted_hidden = (
            transition_model(
                true_prefix_hidden,
                token_1_embedding,
            )
        )

        predicted_logits = (
            target_model.lm_head(
                predicted_hidden
            )
        )

        predicted_probabilities = (
            torch.softmax(
                predicted_logits,
                dim=-1,
            )
        )

        predicted_log_probabilities = (
            torch.log(
                predicted_probabilities
                .clamp_min(1e-12)
            )
        )

        kl = torch.sum(
            p2
            * (
                log_p2
                - predicted_log_probabilities
            ),
            dim=-1,
        )

    return kl.item()

In [21]:
one_step_kl = compute_step_2_kl(
    one_step_model
)

multistep_kl = compute_step_2_kl(
    multistep_model
)

print("One-step model KL:", one_step_kl)
print("Multi-step model KL:", multistep_kl)

One-step model KL: 1.5834081172943115
Multi-step model KL: 1.4036126136779785
